# 3. Traduction automatique — français vers anglais

**Objectif.** Apprendre une correspondance lexicale sur un corpus parallèle contrôlé, puis traduire des combinaisons de mots inédites. Le modèle est un traducteur mot à mot aligné, pas un système général de traduction. Aucune connexion ni API n'est requise.

## 1. Corpus parallèle et séparation

Les phrases suivent la grammaire `pronom + verbe + le + nom` des deux côtés. Chaque mot français est aligné par position sur un mot anglais. Le test contient des **phrases** jamais vues, mais pas nécessairement des mots nouveaux.

In [1]:
from itertools import product
from sklearn.model_selection import train_test_split

subjects = [("je", "i"), ("tu", "you"), ("nous", "we"), ("ils", "they")]
verbs = [
    [("regarde", "watch"), ("regardes", "watch"), ("regardons", "watch"), ("regardent", "watch")],
    [("vois", "see"), ("vois", "see"), ("voyons", "see"), ("voient", "see")],
    [("aime", "like"), ("aimes", "like"), ("aimons", "like"), ("aiment", "like")],
    [("cherche", "seek"), ("cherches", "seek"), ("cherchons", "seek"), ("cherchent", "seek")],
]
nouns = [("chat", "cat"), ("chien", "dog"), ("livre", "book"),
         ("film", "movie"), ("jardin", "garden"), ("train", "train")]
pairs = []
for subject_index, verb_index, noun_index in product(range(4), range(4), range(6)):
    fr_subject, en_subject = subjects[subject_index]
    fr_verb, en_verb = verbs[verb_index][subject_index]
    fr_noun, en_noun = nouns[noun_index]
    pairs.append((f"{fr_subject} {fr_verb} le {fr_noun}",
                  f"{en_subject} {en_verb} the {en_noun}"))
assert len(pairs) == len(set(fr for fr, _ in pairs)) == 96
train_pairs, test_pairs = train_test_split(pairs, test_size=0.25, random_state=42)
assert not set(fr for fr, _ in train_pairs) & set(fr for fr, _ in test_pairs)
print(f"Paires : {len(pairs)} ; train : {len(train_pairs)} ; test : {len(test_pairs)}")
print("Exemple :", train_pairs[0])

Paires : 96 ; train : 72 ; test : 24
Exemple : ('ils aiment le jardin', 'they like the garden')


## 2. Apprentissage du lexique

Chaque paire de jetons alignés fournit une observation. Le mot anglais le plus fréquent est choisi. Un mot inconnu est rendu comme `<UNK>` afin de rendre l'échec explicite.

In [2]:
from collections import Counter, defaultdict

counts = defaultdict(Counter)
for french, english in train_pairs:
    source, target = french.split(), english.split()
    assert len(source) == len(target)
    for fr_word, en_word in zip(source, target):
        counts[fr_word][en_word] += 1
lexicon = {word: frequencies.most_common(1)[0][0] for word, frequencies in counts.items()}

def translate(sentence):
    return " ".join(lexicon.get(word, "<UNK>") for word in sentence.lower().split())

print("Taille du lexique :", len(lexicon))
print("Quelques entrées :", {word: lexicon[word] for word in ["je", "regarde", "le", "chat"]})

Taille du lexique : 26
Quelques entrées : {'je': 'i', 'regarde': 'watch', 'le': 'the', 'chat': 'cat'}


## 3. Exact match et BLEU-4 sur le test

`Exact match` exige une phrase entière identique. BLEU-4 est calculé au niveau du corpus, avec précision des n-grammes 1 à 4 et pénalité de longueur. Le test est comparé à une référence naïve qui recopie le français.

In [3]:
from collections import Counter
from math import exp, log

def corpus_bleu(references, hypotheses, max_n=4):
    assert len(references) == len(hypotheses) and references
    matches = [0] * max_n
    totals = [0] * max_n
    reference_len = hypothesis_len = 0
    for reference, hypothesis in zip(references, hypotheses):
        ref, hyp = reference.split(), hypothesis.split()
        reference_len += len(ref)
        hypothesis_len += len(hyp)
        for n in range(1, max_n + 1):
            ref_ngrams = Counter(tuple(ref[i:i+n]) for i in range(len(ref)-n+1))
            hyp_ngrams = Counter(tuple(hyp[i:i+n]) for i in range(len(hyp)-n+1))
            matches[n-1] += sum(min(count, ref_ngrams[ngram]) for ngram, count in hyp_ngrams.items())
            totals[n-1] += sum(hyp_ngrams.values())
    if hypothesis_len == 0 or any(m == 0 or t == 0 for m, t in zip(matches, totals)):
        return 0.0
    brevity = min(1.0, exp(1 - reference_len / hypothesis_len))
    return 100 * brevity * exp(sum(log(m / t) for m, t in zip(matches, totals)) / max_n)

def evaluate(pairs, system):
    refs = [english for _, english in pairs]
    hyps = [system(french) for french, _ in pairs]
    return {"exact_match": round(sum(h == r for h, r in zip(hyps, refs)) / len(refs), 3),
            "BLEU_4": round(corpus_bleu(refs, hyps), 2)}

print("Copie du français :", evaluate(test_pairs, lambda x: x))
print("Lexique appris    :", evaluate(test_pairs, translate))
print("Exemples du test :")
for french, english in test_pairs[:5]:
    print(f"{french} -> {translate(french)} | référence : {english}")

Copie du français : {'exact_match': 0.0, 'BLEU_4': 0.0}
Lexique appris    : {'exact_match': 1.0, 'BLEU_4': 100.0}
Exemples du test :
ils voient le livre -> they see the book | référence : they see the book
ils regardent le train -> they watch the train | référence : they watch the train
ils regardent le chien -> they watch the dog | référence : they watch the dog
ils cherchent le jardin -> they seek the garden | référence : they seek the garden
tu vois le film -> you see the movie | référence : you see the movie


## 4. Test hors grammaire

Ce petit défi montre ce qui se passe avec un déterminant nouveau, un pluriel et une structure grammaticale non vue. Les métriques sont séparées du test principal.

In [4]:
challenge = [
    ("je regarde un film", "i watch a movie"),
    ("nous aimons les livres", "we like the books"),
    ("le chat regarde le film", "the cat watches the movie"),
]
print("Défi hors grammaire :", evaluate(challenge, translate))
for french, english in challenge:
    print(f"{french} -> {translate(french)} | référence : {english}")

Défi hors grammaire : {'exact_match': 0.0, 'BLEU_4': 0.0}
je regarde un film -> i watch <UNK> movie | référence : i watch a movie
nous aimons les livres -> we like <UNK> <UNK> | référence : we like the books
le chat regarde le film -> the cat watch the movie | référence : the cat watches the movie


## 5. Limites

Les paires sont générées par une grammaire artificielle et l'alignement est fourni par construction. Un BLEU élevé dans ce cadre ne démontre pas une traduction correcte en situation réelle. La méthode ne gère pas l'ordre des mots, la flexion, les mots inconnus ni les sens multiples. Pour un vrai système, utiliser un corpus parallèle indépendant et un modèle de traduction adapté, puis compléter BLEU par une évaluation humaine.